In [72]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

ds_name = "mnist"
split = "trainUval"

epoch = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
dses = find_all_datasets("../../datasets/")
ds = dses[ds_name]

In [74]:
data_dir = "C:/home/ae_data/landscape_data"
strees_dir = "C:/home/ae_data/strees_ae/"

In [75]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[0]
for e in exps:
	if e.dataset.name == ds_name and e.split == split and e.epoch == epoch:
		exp = e
		break

assert exp.dataset.name == ds_name
assert exp.split == split
assert exp.epoch == epoch

e

mnist (mnist-trainUval), k=20, layer=latents, epoch=100

In [76]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [77]:
import pyct as ct

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

homo_prop = 0.99
fns, counts = simpl.getHomoValleyPlot(order, wts, labels, homo_prop, partition) # type: ignore
fns_norm, counts_norm_min, counts_norm_max = simpl.getSimplificationPlot(order, wts)

In [78]:
def count_stability(valleys, fns):    
	stabilities = {}
	
	for i, fn in enumerate(fns[1:]):
		i += 1
		
		val = valleys[i]
		if val < valleys[i - 1]:
			stabilities[valleys[i - 1]] = (fn - fns[i - 1], fns[i - 1])

	stabilities[valleys[-1]] = (fns[-1], fns[-1])
	
	return stabilities
		

In [108]:
import plotly.express as px

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=fns, y=counts, mode='lines', name=f'{homo_prop*100:.1f}% Valleys', line=dict(color='blue', shape="hv")))
fig.add_trace(go.Scatter(x=fns_norm, y=counts_norm_min, mode='lines', name='Valleys', line=dict(color='red', shape="hv")))

fig.show()

In [80]:
import pandas as pd

stab_df = pd.DataFrame([(k, v, last_value) for k, (v, last_value) in count_stability(counts, fns).items()], columns=["hvalleys", "stability", "fn_end"])
stab_df.sort_values(by=["stability"], ascending=False).head(10)

,hvalleys,stability,fn_end
1322,0,0.028543,0.028543
1320,2,0.005874,0.020424
1321,1,0.002244,0.026299
1319,3,0.001508,0.018161
1317,5,0.000681,0.016580
1313,10,0.000523,0.015432
1308,15,0.000419,0.014204
1295,28,0.000410,0.012307
1316,6,0.000389,0.016191
1318,4,0.000368,0.017260


In [82]:
import pandas as pd

stab_df = pd.DataFrame([(k, v, last_value) for k, (v, last_value) in count_stability(counts_norm_min, fns_norm).items()], columns=["valleys", "stability", "fn_end"])
stab_df.sort_values(by=["stability"], ascending=False).head(10)

,valleys,stability,fn_end
1351,1,0.028543,0.028543
1349,3,0.005874,0.020424
1350,2,0.002244,0.026299
1347,5,0.001508,0.018161
1348,4,0.000755,0.019669
1344,8,0.000681,0.016580
1346,6,0.000533,0.017628
1340,12,0.000523,0.015432
1335,17,0.000419,0.014204
1322,30,0.000410,0.012307


In [83]:
data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

fns_more, remaining_all, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = simpl.getHomoValleyPlotPlusCoverages(order, wts, labels, homo_prop, partition)

In [84]:
classes_needed = 9 # 10 classes, the following criteria must apply to this many of them
maj_coverage_needed = 0.01 # at least 1% coverage in homogeneous valleys where they are the only class (100% proportion in valley) 
total_coverage_needed = 0.02 # at least 5% total coverage in all valleys

# coverages are functions of accuracy, so this constraint necessarily tightens in more complex datasets

represented = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) >= classes_needed 
            	for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]

# find interval where this is true
last_idx = len(fns) - represented[::-1].index(True) - 1

first_idx = represented.index(True)

print(f"Uniform Interval: {all(represented[first_idx:last_idx+1])}")
print(f"First IDX: {first_idx} - ")
print(first_idx, len(fns), fns[first_idx], list(zip(maj_class_homo_cov[first_idx], class_all_covs[first_idx], maj_class_homo_counts[first_idx])), sep="\n")
print(f"\nLast IDX: {last_idx} - ")
print(last_idx, len(fns), fns[last_idx], list(zip(maj_class_homo_cov[last_idx], class_all_covs[last_idx], maj_class_homo_counts[last_idx])), sep="\n")

Uniform Interval: True
First IDX: 0 - 
0
1352
0.0
[(0.04592206287121541, 0.046646385629436474, 185), (0.05217722483178875, 0.05255808048749524, 213), (0.026466380543633764, 0.026609442060085836, 118), (0.02506651729449657, 0.025906735751295335, 119), (0.03179953106682298, 0.032825322391559206, 126), (0.02471091398701093, 0.02645335022968478, 99), (0.028068644560791157, 0.029668411867364748, 126), (0.0337309749074455, 0.03427944604415192, 146), (0.01597069597069597, 0.01641025641025641, 80), (0.024576027594136247, 0.02644438056912906, 112)]

Last IDX: 289 - 
289
1352
0.0009588822722434998
[(0.04244531363175431, 0.043169636389975376, 162), (0.0449409673733655, 0.045321823029071985, 154), (0.02188841201716738, 0.022317596566523604, 93), (0.020305279372636886, 0.02100546141996919, 90), (0.027549824150058615, 0.02857561547479484, 102), (0.02059242832250911, 0.022176461270394424, 78), (0.02326934264107039, 0.02472367655613729, 102), (0.029617441382147263, 0.030028794734677088, 116), (0.01260

In [85]:
remaining_all[last_idx]

1063